#### A Streamlit driven UI to upload an Image and derive the caption

#### Required Packages + Models

1. Streamlit
2. ngrok
3. tensorflow + transformers
4. model weights .h5
5. tokenizer.pkl
6. Custom Decoder Model (used earlier)
   


#### Steps to Generate Caption

1. Run the ngrok server
2. Run the streamlit app.py
3. Upload Image
4. Generate Caption

#### Quick Internals

1. Feature Extraction + Patch Embedding with Vision Transformer Model
2. Load Custom Model + Tokenizer
3. Generate Caption for the uploaded Image

In [99]:
!pip install -q streamlit

In [100]:
!pip install tensorflow transformers 

In [101]:
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.9.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 2.8.0 requires google-cloud-bigquery[bqstorage,pandas]>=3.31.0, but you have google-cloud-bigquery 3.25.0 which is incompatible.
bigframes 2.8.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.


#### Installing ngrok , pyngrok

In [102]:
pip install pyngrok

Note: you may need to restart the kernel to use updated packages.


In [103]:
!pip install --no-dependencies --quiet streamlit
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
!unzip -o ngrok-stable-linux-amd64.zip
!pip install --quiet pyngrok
!pip install --no-dependencies --quiet protobuf==3.20.*   #==4.21.12
!pip install --no-dependencies --quiet validators


--2025-08-15 11:40:59--  https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 75.2.60.68, 35.71.179.82, 99.83.220.108, ...
Connecting to bin.equinox.io (bin.equinox.io)|75.2.60.68|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13921656 (13M) [application/octet-stream]
Saving to: ‘ngrok-stable-linux-amd64.zip.2’

ngrok-stable-linux- 100%[===================>]  13.28M  48.7MB/s    in 0.3s    

2025-08-15 11:40:59 (48.7 MB/s) - ‘ngrok-stable-linux-amd64.zip.2’ saved [13921656/13921656]

Archive:  ngrok-stable-linux-amd64.zip
  inflating: ngrok                   


#### App.py file  
1. The file will be created in /kaggle/working dir
2. We will be running this file via Streamlit inside ngrok

In [118]:
%%writefile app.py

import streamlit as st
import numpy as np
from PIL import Image
import tensorflow as tf
from transformers import ViTFeatureExtractor, TFAutoModel
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, AdditiveAttention
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
import pickle
import h5py
from gtts import gTTS
import os

# Decoder model
class CaptionDecoder(Model):
    def __init__(self, vocab_size, embed_dim=256, lstm_units=512,**kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.lstm_units = lstm_units
        
        self.embedding = Embedding(vocab_size, embed_dim)
        self.attention = AdditiveAttention()
        self.lstm = LSTM(lstm_units, return_sequences=True)
        self.fc = Dense(vocab_size)
        self.attn_weights = None
        self.feature_proj = Dense(embed_dim)

    def call(self, features, captions):
        features = self.feature_proj(features)  # [B, 196, embed_dim]
        embedded = self.embedding(captions)     # [B, T, embed_dim]
        context = self.attention([embedded, features],return_attention_scores=True)  # [B, T, embed_dim]
        self.attn_weights = context[1]  # Save for viz
        x = tf.concat([context[0], embedded], axis=-1)
        x = self.lstm(x)
        return self.fc(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim,
            "lstm_units": self.lstm_units
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

# Load HF ViT model and feature extractor
feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k")
vit_model = TFAutoModel.from_pretrained("google/vit-base-patch16-224-in21k")

def verify_model_weights():
    with h5py.File("/kaggle/input/model1/transformers/default/1/caption_decoder.weights.h5", "r") as f:
        return list(f.keys())

def retrieve_tokenizer():        
    file_path = '/kaggle/input/model1/transformers/default/1/tokenizer.pickle'
    with open(file_path, 'rb') as file:
        tokenizer = pickle.load(file)
    return tokenizer

# Image preprocessing + feature extraction
def extract_features(img):
    inputs = feature_extractor(images=img, return_tensors="tf")
    outputs = vit_model(inputs["pixel_values"])
    patch_embeddings = outputs.last_hidden_state[:, 1:, :]  # Remove [CLS] token → shape: [1, 196, 768]
    return tf.squeeze(patch_embeddings, axis=0)  # shape: [196, 768]

def load_model(vocab_size):
    h5_model = CaptionDecoder(vocab_size=vocab_size)
    dummy_features = tf.random.uniform((1, 196, 768))
    dummy_captions = tf.random.uniform((1, 20), maxval=vocab_size, dtype=tf.int32)
    h5_model(dummy_features,dummy_captions)
    h5_model.load_weights('/kaggle/input/model1/transformers/default/1/caption_decoder.weights.h5')
    return h5_model

def generate_caption(features,model,tokenizer,max_len=30):
    input_seq = tokenizer.texts_to_sequences(["<start>"])[0]
    for _ in range(max_len):
        seq = pad_sequences([input_seq], maxlen=max_len, padding='post')
        preds = model(tf.expand_dims(features, 0), tf.convert_to_tensor(seq))
        pred_id = tf.argmax(preds[0, len(input_seq)-1]).numpy()
        index_word = {v: k for k, v in tokenizer.word_index.items()}
        word = index_word.get(pred_id, '<unk>')
        if word == '<end>':
            break
        input_seq.append(pred_id)

    return ' '.join([index_word.get(i, '') for i in input_seq[1:]])

def play_audio(caption):
    caption_audio=gTTS(text=caption,lang='en',slow=False)
    caption_audio.save("output.mp3")
    with open("output.mp3", "rb") as f:
        audio_bytes = f.read()
    st.audio(audio_bytes, format="audio/mp3")

with st.container():
    st.title("Image Caption Reader")
    st.write("Upload an image and get predictions.")
    uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "png", "jpeg"])

    if uploaded_file is not None:
        # Display image
        image = Image.open(uploaded_file).convert("RGB")
        st.image(image, caption="Uploaded Image", use_column_width=True)

        # Preprocess (adjust as per your model's input shape)
        img_resized = image.resize((224, 224))

        features=extract_features(img_resized)
        tokenizer=retrieve_tokenizer()
        vocab_size = len(tokenizer.word_index) + 1
        st.write('Model Weights : '+str(verify_model_weights()))
        st.write('vocab_size : '+str(vocab_size))
        model=load_model(vocab_size=vocab_size)
        caption=generate_caption(features,model,tokenizer)
        st.success(str(caption))
        play_audio(str(caption))

Overwriting app.py


#### Saving ngrok Auth Token

In [105]:
!ngrok authtoken "31JaQiosRxacYtnKuZwlpNgrEtP_QhGHk5koxR3KR5CyU4kc"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [106]:
!cat /root/.config/ngrok/ngrok.yml

region: us
version: '2'
authtoken: 31JaQiosRxacYtnKuZwlpNgrEtP_QhGHk5koxR3KR5CyU4kc


#### Running the app.py with streamlit under ngrok tunnel

In [119]:
import os
import threading
import time
import subprocess

# Make Streamlit config
os.makedirs(os.path.expanduser("~/.streamlit"), exist_ok=True)
with open(os.path.expanduser("~/.streamlit/config.toml"), "w") as f:
    f.write("[server]\nheadless = true\nenableCORS = false\nenableXsrfProtection = false\n")

# Start Ngrok tunnel
public_url = ngrok.connect(8501)
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://603c165b5077.ngrok-free.app" -> "http://localhost:8501"


In [120]:

# Start Streamlit in a thread
def run_streamlit():
    subprocess.run(["streamlit", "run", "app.py", "--server.port", "8501"])

threading.Thread(target=run_streamlit, daemon=True).start()

# Keep notebook cell alive
while True:
    time.sleep(60)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://35.196.0.142:8501



2025-08-15 12:12:51.070093: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755259971.103834    1305 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755259971.113774    1305 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
2025-08-15 12:13:02.425292: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (30

  Stopping...


KeyboardInterrupt: 